In [33]:
import pandas as pd
import numpy as np
from cpp_poker.cpp_poker import Hand

In [34]:
df = pd.read_parquet("../dfs/compacted_20240430132106.parquet")
df

,prob_P_has_hand_0,prob_P_has_hand_1,prob_P_has_hand_2,prob_P_has_hand_3,prob_P_has_hand_4,prob_P_has_hand_5,prob_P_has_hand_6,prob_P_has_hand_7,prob_P_has_hand_8,prob_P_has_hand_9,...,player_bet_in_stage,player_bet_in_game,opponent_bet_in_stage,opponent_bet_in_game,player_turn,player_has_bet,opponent_has_bet,pot,game_size,stage
0,0.001377,0.000228,0.001744,0.000486,0.000538,0.000434,0.000950,0.001289,0.000914,0.001696,...,14,469,40,495,True,True,True,964,1487,river
1,0.001339,0.000039,0.000013,0.000504,0.000867,0.001299,0.001545,0.000667,0.001549,0.000661,...,11,119,13,121,True,True,True,240,2003,river
2,0.000000,0.001377,0.000651,0.000391,0.001705,0.000569,0.000410,0.001504,0.000090,0.000198,...,110,370,117,377,True,True,True,747,2001,river
3,0.001162,0.001362,0.000000,0.001594,0.001635,0.001112,0.000135,0.001304,0.000315,0.001661,...,0,305,325,630,True,True,True,935,2326,river
4,0.000171,0.000427,0.001027,0.000752,0.000495,0.000646,0.001213,0.000560,0.001425,0.001687,...,0,180,28,208,True,False,True,388,1088,river
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11007,0.001510,0.001131,0.001499,0.000144,0.000465,0.000000,0.000973,0.001789,0.001542,0.001787,...,0,103,45,148,True,False,True,251,1333,river
11008,0.000990,0.001654,0.000506,0.000000,0.000199,0.001468,0.001385,0.001262,0.001167,0.000688,...,0,409,92,501,True,False,True,910,1907,river
11009,0.000293,0.000851,0.001572,0.001028,0.000965,0.001796,0.000691,0.001638,0.001164,0.000123,...,0,143,167,310,True,True,True,453,1442,river
11010,0.001447,0.000098,0.000460,0.000000,0.000387,0.000091,0.001042,0.000552,0.001136,0.000000,...,0,439,118,557,True,False,True,996,2194,river


In [35]:
card_cols = [col for col in df.columns if col.startswith("public_card")]
df[card_cols]

,public_card_0,public_card_1,public_card_2,public_card_3,public_card_4,public_card_5,public_card_6,public_card_7,public_card_8,public_card_9,...,public_card_42,public_card_43,public_card_44,public_card_45,public_card_46,public_card_47,public_card_48,public_card_49,public_card_50,public_card_51
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11007,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
11008,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
11010,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [36]:
assert df[card_cols].sum(axis=1).eq(5).all()

In [37]:
# Check that the error exists
for row in range(df.shape[0]):
    for card, col in enumerate(card_cols):
        if df.loc[row, col] == 1:
            relevant_range_cols = [
                f"prob_P_has_hand_{conf_hand}" for conf_hand in Hand.HANDS_WITH_CARD[card]
            ] + [f"prob_O_has_hand_{conf_hand}" for conf_hand in Hand.HANDS_WITH_CARD[card]]
            range_sum = df.loc[row, relevant_range_cols].sum()
            if range_sum != 0:
                print("Row:", row, "Card:", card, "conflicts with hands")
                print("Row:", row, "Range sum:", range_sum)
                print("Row:", row, df.loc[row, relevant_range_cols])
                break
    else:
        continue
    break

Row: 260 Card: 10 conflicts with hands
Row: 260 Range sum: 0.07603216268205765
Row: 260 prob_P_has_hand_9      0.000643
prob_P_has_hand_59     0.000031
prob_P_has_hand_108    0.000374
prob_P_has_hand_156    0.001243
prob_P_has_hand_203    0.001456
                         ...   
prob_O_has_hand_501    0.000513
prob_O_has_hand_502    0.001156
prob_O_has_hand_503    0.000693
prob_O_has_hand_504    0.000877
prob_O_has_hand_505    0.001218
Name: 260, Length: 102, dtype: object


In [38]:
# Fix the error
p1_range_cols = [col for col in df.columns if col.startswith("prob_P_has_hand_")]
opponent_range_cols = [col for col in df.columns if col.startswith("prob_O_has_hand_")]
updated_rows = set()
range_cols_per_card = np.array(
    [
        np.array(
            [f"prob_P_has_hand_{conf_hand}" for conf_hand in Hand.HANDS_WITH_CARD[card]]
            + [
                f"prob_O_has_hand_{conf_hand}"
                for conf_hand in Hand.HANDS_WITH_CARD[card]
            ]
        )
        for card in range(52)
    ]
)
for row in range(df.shape[0]):
    if row % 100 == 0:
        print("Row:", row, end="\r")
    relevant_range_cols = range_cols_per_card[df.loc[row, card_cols] == 1]
    updated_row = False
    for cols in relevant_range_cols:
        range_sum = df.loc[row, cols].sum()
        if range_sum != 0:
            df.loc[row, cols] = 0
            updated_row = True
    if updated_row:
        print("Row:", row, "Fixed")
        updated_rows.add(row)
print("Fixed", len(updated_rows), "rows")

Row: 260 Fixed
Row: 661 Fixed
Row: 662 Fixed
Row: 663 Fixed
Row: 664 Fixed
Row: 665 Fixed
Row: 666 Fixed
Row: 667 Fixed
Row: 668 Fixed
Row: 669 Fixed
Row: 670 Fixed
Row: 671 Fixed
Row: 672 Fixed
Row: 673 Fixed
Row: 674 Fixed
Row: 675 Fixed
Row: 676 Fixed
Row: 677 Fixed
Row: 678 Fixed
Row: 679 Fixed
Row: 680 Fixed
Row: 681 Fixed
Row: 682 Fixed
Row: 683 Fixed
Row: 684 Fixed
Row: 685 Fixed
Row: 686 Fixed
Row: 687 Fixed
Row: 688 Fixed
Row: 689 Fixed
Row: 690 Fixed
Row: 691 Fixed
Row: 692 Fixed
Row: 693 Fixed
Row: 694 Fixed
Row: 695 Fixed
Row: 696 Fixed
Row: 697 Fixed
Row: 698 Fixed
Row: 699 Fixed
Row: 700 Fixed
Row: 701 Fixed
Row: 702 Fixed
Row: 703 Fixed
Row: 704 Fixed
Row: 705 Fixed
Row: 706 Fixed
Row: 707 Fixed
Row: 708 Fixed
Row: 709 Fixed
Row: 710 Fixed
Row: 711 Fixed
Row: 712 Fixed
Row: 713 Fixed
Row: 714 Fixed
Row: 715 Fixed
Row: 716 Fixed
Row: 717 Fixed
Row: 718 Fixed
Row: 719 Fixed
Row: 720 Fixed
Row: 721 Fixed
Row: 722 Fixed
Row: 723 Fixed
Row: 724 Fixed
Row: 725 Fixed
Row: 726 F

In [39]:
# Check that the error no longer exists
for row in range(df.shape[0]):
    for card, col in enumerate(card_cols):
        if df.loc[row, col] == 1:
            relevant_range_cols = [
                f"prob_P_has_hand_{conf_hand}" for conf_hand in Hand.HANDS_WITH_CARD[card]
            ] + [f"prob_O_has_hand_{conf_hand}" for conf_hand in Hand.HANDS_WITH_CARD[card]]
            range_sum = df.loc[row, relevant_range_cols].sum()
            if range_sum != 0:
                print("Row:", row, "Card:", card, "conflicts with hands")
                print("Row:", row, "Range sum:", range_sum)
                print("Row:", row, df.loc[row, relevant_range_cols])
                raise Exception("Error still exists")
print("No error in any rows!")

In [40]:
# Make ranges sum to 1
p1_range_cols = [col for col in df.columns if col.startswith("prob_P_has_hand_")]
opponent_range_cols = [col for col in df.columns if col.startswith("prob_O_has_hand_")]
updated_rows = list(updated_rows)
df.loc[updated_rows, p1_range_cols] = df.loc[updated_rows, p1_range_cols].div(
    df.loc[updated_rows, p1_range_cols].sum(axis=1), axis=0
)
df.loc[updated_rows, opponent_range_cols] = df.loc[updated_rows, opponent_range_cols].div(
    df.loc[updated_rows, opponent_range_cols].sum(axis=1), axis=0
)

In [44]:
assert np.allclose(df[p1_range_cols].sum(axis=1), 1, rtol=1e-10)
assert np.allclose(df[opponent_range_cols].sum(axis=1), 1, rtol=1e-10)

In [46]:
# Check that a similar error does not exist for the target variables
for row in range(df.shape[0]):
    for card, col in enumerate(card_cols):
        if df.loc[row, col] == 1:
            relevant_target_cols = [
                f"value_of_hand_{conf_hand}" for conf_hand in Hand.HANDS_WITH_CARD[card]
            ]
            range_sum = df.loc[row, relevant_target_cols].sum()
            if range_sum != 0:
                print("Row:", row, "Card:", card, "conflicts with hands")
                print("Row:", row, "Values for conflicting hands:", range_sum)
                print("Row:", row, df.loc[row, relevant_target_cols])
                raise Exception("The same error exists for the target variables")
print("No error!")

No error!


In [47]:
# Add a column to distinguish between fixed and non-fixed rows
df.loc[:, "origin"] = "after_range_fix"
df.loc[updated_rows, "origin"] = "pre_range_fix_but_fixed"
df

,prob_P_has_hand_0,prob_P_has_hand_1,prob_P_has_hand_2,prob_P_has_hand_3,prob_P_has_hand_4,prob_P_has_hand_5,prob_P_has_hand_6,prob_P_has_hand_7,prob_P_has_hand_8,prob_P_has_hand_9,...,player_bet_in_game,opponent_bet_in_stage,opponent_bet_in_game,player_turn,player_has_bet,opponent_has_bet,pot,game_size,stage,origin
0,0.001377,0.000228,0.001744,0.000486,0.000538,0.000434,0.000950,0.001289,0.000914,0.001696,...,469,40,495,True,True,True,964,1487,river,after_range_fix
1,0.001339,0.000039,0.000013,0.000504,0.000867,0.001299,0.001545,0.000667,0.001549,0.000661,...,119,13,121,True,True,True,240,2003,river,after_range_fix
2,0.000000,0.001377,0.000651,0.000391,0.001705,0.000569,0.000410,0.001504,0.000090,0.000198,...,370,117,377,True,True,True,747,2001,river,after_range_fix
3,0.001162,0.001362,0.000000,0.001594,0.001635,0.001112,0.000135,0.001304,0.000315,0.001661,...,305,325,630,True,True,True,935,2326,river,after_range_fix
4,0.000171,0.000427,0.001027,0.000752,0.000495,0.000646,0.001213,0.000560,0.001425,0.001687,...,180,28,208,True,False,True,388,1088,river,after_range_fix
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11007,0.001510,0.001131,0.001499,0.000144,0.000465,0.000000,0.000973,0.001789,0.001542,0.001787,...,103,45,148,True,False,True,251,1333,river,after_range_fix
11008,0.000990,0.001654,0.000506,0.000000,0.000199,0.001468,0.001385,0.001262,0.001167,0.000688,...,409,92,501,True,False,True,910,1907,river,after_range_fix
11009,0.000293,0.000851,0.001572,0.001028,0.000965,0.001796,0.000691,0.001638,0.001164,0.000123,...,143,167,310,True,True,True,453,1442,river,after_range_fix
11010,0.001447,0.000098,0.000460,0.000000,0.000387,0.000091,0.001042,0.000552,0.001136,0.000000,...,439,118,557,True,False,True,996,2194,river,after_range_fix


In [48]:
# Save fixed dataframe to parquet
df.to_parquet("../dfs/fixed_20240430132106.parquet")